<a href="https://colab.research.google.com/github/Gidayi-dev/Dummy/blob/main/tzTourism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold

In [16]:
train = pd.read_csv("Train.csv")
test = pd.read_csv("Test.csv")
sample = pd.read_csv("SampleSubmission.csv")

print(f"Train: {train.shape}, Test: {test.shape}")
print("\nMissing values:\n", train.isnull().sum()[train.isnull().sum() > 0])
print(f"\nTarget skewness: {train['total_cost'].skew():.2f}")
print(f"After log1p: {np.log1p(train['total_cost']).skew():.2f}")

Train: (4809, 23), Test: (1601, 22)

Missing values:
 travel_with        1114
total_female          3
total_male            5
most_impressing     313
dtype: int64

Target skewness: 2.97
After log1p: -0.32


In [17]:
from numpy._core.defchararray import lower
def engineer(df):
  df = df.copy()
  df['total_nights'] = df['night_mainland'] + df['night_zanzibar']
  df['group_size'] = (df['total_female'].fillna(0) + df['total_male'].fillna(0)).clip(lower=1)
  df['zanzibar_ratio'] = df['night_zanzibar'] / (df['total_nights'] + 1)

  pkg_cols = [
      'package_transport_int', 'package_accomodation', 'package_food',
      'package_transport_tz', 'package_sightseeing', 'package_guided_tour',
      'package_insurance'
  ]
  for c in pkg_cols:
    df[c + '_bin'] = (df[c] == 'Yes').astype(int)
  df['package_score'] = df[[c + '_bin' for c in pkg_cols]].sum(axis=1)
  df['is_package_tour'] = (df['tour_arrangement'] == 'Packaghe Tour').astype(int)
  df['is_first_trip'] = (df['first_trip_tz'] == 'Yes').astype(int)

  return df

train = engineer(train)
test = engineer(test)


In [18]:
from pandas.core.arrays import categorical
numeric_features = [
    'total_female', 'total_male', 'night_mainland', 'night_zanzibar',
    'total_nights', 'group_size', 'zanzibar_ratio', 'package_score', 'is_package_tour', 'is_first_trip',
    'package_transport_int_bin', 'package_accomodation_bin', 'package_food_bin',
    'package_transport_tz_bin', 'package_sightseeing_bin',
    'package_guided_tour_bin', 'package_insurance_bin'
]

categorical_features = [
    'country', 'age_group', 'travel_with', 'purpose', 'main_activity',
    'info_source', 'tour_arrangement', 'payment_mode', 'first_trip_tz'
]
x = train[numeric_features + categorical_features]
y = np.log1p(train['total_cost'])
x_test = test[numeric_features + categorical_features]

In [19]:
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', cat_pipe, categorical_features)
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42
    ))
])

In [20]:
print("\n=== 5-Fold Cross Validation ===")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_maes = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(x)):
  x_tr, x_val = x.iloc[tr_idx], x.iloc[val_idx]
  y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
  model.fit(x_tr, y_tr)
  preds_val = np.expm1(model.predict(x_val))
  actual = np.expm1(y_val)

  mae = np.mean(np.abs(actual - preds_val))
  fold_maes.append(mae)
  print(f" Fold {fold+1}: MAE = {mae:,.0f} TZS")

print(f"\nMean MAE: {np.mean(fold_maes):,.0f} TZS")


=== 5-Fold Cross Validation ===
 Fold 1: MAE = 4,703,977 TZS
 Fold 2: MAE = 4,519,529 TZS
 Fold 3: MAE = 5,480,031 TZS
 Fold 4: MAE = 4,697,193 TZS
 Fold 5: MAE = 5,163,420 TZS

Mean MAE: 4,912,830 TZS


In [21]:
print("\nTraining on full dataset...")
model.fit(x, y)
preds = np.expm1(model.predict(x_test))
print(f"predictions - min: {preds.min():,.0f} max: {preds.max():,.0f} mean: {preds.mean():,.0f}")


Training on full dataset...
predictions - min: 86,556 max: 42,556,575 mean: 5,467,198


In [22]:
submission = pd.DataFrame({'ID': test['ID'], 'total_cost': preds})
submission.to_csv('submission.csv', index=False)
print(submission.head())

          ID    total_cost
0     tour_1  1.496102e+07
1   tour_100  3.281116e+06
2  tour_1001  1.075647e+07
3  tour_1006  1.193124e+06
4  tour_1009  1.841929e+07


In [23]:
from google.colab import files
files.download('submission.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>